In [2]:
!pip install -q pycolmap torchmetrics gsplat
!pip install -q git+https://github.com/compphoto/Intrinsic.git
!pip install -q opencv-python

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import os

try:
    from google.colab import drive
    os.chdir('/content')
    drive.mount('/content/drive', force_remount=True)

    !pip install -q git+https://github.com/compphoto/Intrinsic.git
    !pip install opencv-python
    os.chdir('/content/drive/MyDrive/3dgs_decomposition')
except ModuleNotFoundError:
    pass  # running locally, already in project root

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:

import argparse, glob, os
import numpy as np, torch
import torch.nn.functional as F
from PIL import Image
from gsplat import rasterization
from gsplat.strategy import DefaultStrategy
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
import os

dev = "cuda"

def get_image_paths(root):
  img_dir = os.path.join(root, "images_8")
  paths = []
  for path in os.listdir(img_dir):
    if (path.endswith(".JPG") || path.endswith(".jpg")):
      paths.append(os.path.join(img_dir, path))

  return paths

import pycolmap

def getw2c(image):
    cam_from_world = image.cam_from_world
    if callable(cam_from_world):
        cam_from_world = cam_from_world()

    w2c = np.eye(4)
    w2c[:3, :4] = np.asarray(cam_from_world.matrix(), dtype=np.float64)
    return w2c

def get_cameras(root, paths):
  colmap_dir = os.path.join(root, "sparse/0")

  reconstruction = pycolmap.Reconstruction(colmap_dir)
  cameras = _as_dict(reconstruction.cameras)

  w2c_mats = []
  camera_ids = []
  by_name = {}

  images = _as_dict(reconstruction.images)
  image_ids = [int(image_id) for image_id in reconstruction.reg_image_ids()]
  imdata = {image_id: images[image_id] for image_id in image_ids}


  for image_id in imdata:
    im = imdata[image_id]
    w2c_mats.append(getw2c(im))
    camera_ids.append(im.camera_id)
    by_name[im.name] = (getw2c(im), int(im.camera_id))


  width, height = Image.open(paths[0]).size
  viewmats = []
  intrinsics = []
  for path in paths:
    name = os.path.basename(path)
    w2c, camera_id = by_name[name]
    cam = cameras[camera_id]
    K = np.asarray(cam.calibration_matrix(), dtype=np.float64).copy()
    K[0, :] *= width / cam.width
    K[1, :] *= height / cam.height

    intrinsics.append(K)
    viewmats.append(w2c)

  points = np.array([p.xyz for p in reconstruction.points3D.values()], dtype=np.float32)

  return (np.stack(viewmats), np.stack(intrinsics), (width, height), points,)


In [5]:
def _as_dict(map_like):
    return {int(k): v for k, v in map_like.items()}

In [ ]:
import pycolmap

ROOT = "/content/drive/MyDrive/3dgs/data/mipnerf360/room"

paths = get_image_paths(ROOT)

viewmats, intrinsics, (width, height), points = get_cameras(ROOT, paths)

In [ ]:
def get_views(paths, root, predictor, dataset, scene):

  images = []
  albedos = []
  depths = []

  for p in paths:
    file_name = os.path.basename(p).split('.')[0]
    print(file_name)
    i = torch.tensor(np.array(Image.open(p).convert("RGB")), dtype=torch.float32) / 255
    width, height = Image.open(p).size


    assert i.ndim == 3 and i.shape[2] == 3, (
            f"Expected (H, W, 3) for {p}, got {tuple(i.shape)}"
        )


    albedo_path = os.path.join(root, predictor, dataset, scene, "albedos", f"{file_name}.npy")
    albedo = torch.tensor(np.load(albedo_path), dtype=torch.float32)
    albedos.append(albedo)

    depth_path = os.path.join(root, "depth_anything_v2", "depths", dataset,
                              scene, f"{file_name}.png",
  )

    depth = torch.tensor(
        np.array(Image.open(depth_path), dtype=np.float32), dtype=torch.float32
    )

    depth_npy_path = os.path.splitext(depth_path)[0] + ".npy"
    np.save(depth_npy_path, depth.detach().cpu().numpy())

    depths.append(depth.detach().clone())
    images.append(i)

  return torch.stack(images).to(dev), torch.stack(albedos).to(dev), torch.stack(depths).to(dev)


In [34]:
def pearson_loss(d, m):
    d, m = d.flatten(), m.flatten()
    d = (d - d.mean()) / (d.std() + 1e-6); m = (m - m.mean()) / (m.std() + 1e-6)
    return 1 - (d * m).mean()

def photo_loss(pred, tgt, lam=0.2):
    if pred.ndim == 4 and pred.shape[0] == 1:
        pred = pred.squeeze(0)

    if tgt.ndim == 4 and tgt.shape[0] == 1:
        tgt = tgt.squeeze(0)
    l1 = (pred - tgt).abs().mean()
    pred = pred.permute(2, 0, 1).unsqueeze(0)
    tgt = tgt.permute(2, 0, 1).unsqueeze(0)

    s = 1 - ssim_fn(pred, tgt)

    return (1 - lam) * l1 + lam * s

def tv(x):
    return (x[1:] - x[:-1]).abs().mean() + (x[:, 1:] - x[:, :-1]).abs().mean()

In [ ]:

learning_rates = {
    "means": 1.6e-4,
    "scales": 5e-3,
    "quats": 1e-3,
    "opacities": 5e-2,
    "colors": 1e-2,
}

def create_gaussians(means, scales, quats, opacities, colors, scene_scale=1.0):
    params = torch.nn.ParameterDict({
        "means": torch.nn.Parameter(means),
        "scales": torch.nn.Parameter(scales),
        "quats": torch.nn.Parameter(quats),
        "opacities": torch.nn.Parameter(opacities),
        "colors": torch.nn.Parameter(colors),
    })
    optimizers = {
        "means": torch.optim.Adam([params["means"]], lr=learning_rates["means"] * scene_scale),
        "scales": torch.optim.Adam([params["scales"]], lr=learning_rates["scales"]),
        "quats": torch.optim.Adam([params["quats"]], lr=learning_rates["quats"]),
        "opacities": torch.optim.Adam([params["opacities"]], lr=learning_rates["opacities"]),
        "colors": torch.optim.Adam([params["colors"]], lr=learning_rates["colors"]),
    }
    return params, optimizers

In [ ]:
from scipy.spatial import cKDTree

def initialise_gaussians(raw_points, init_scale=1.0, init_opacity=0.1, scene_scale=1.0, dev=dev):
    pts_cpu = np.asarray(raw_points, dtype=np.float32)

    tree = cKDTree(pts_cpu)

    distances, _ = tree.query(pts_cpu, k=4)

    average_distance = np.maximum(distances[:, 1:].mean(axis=-1), 1e-4)

    points = torch.as_tensor(pts_cpu, dtype=torch.float32, device=dev)
    average_distance = torch.as_tensor(average_distance, dtype=torch.float32, device=dev)
    N = len(points)

    scales = torch.log(average_distance * init_scale).unsqueeze(-1).repeat(1, 3)
    quats = F.normalize(torch.rand((N, 4), device=dev), dim=-1)
    opacities = torch.logit(torch.full((N,), init_opacity, device=dev))
    colors = torch.zeros((N, 3), device=dev)


    return create_gaussians(points, scales, quats, opacities, colors, scene_scale=scene_scale)

In [9]:
def initialise_from_albedos(checkpoint, color_shape, initial_color=0.0):
  means = checkpoint["means"].to(dev).clone()
  scales = checkpoint["scales"].to(dev).clone()
  quats = checkpoint["quats"].to(dev).clone()
  opacities = checkpoint["opacities"].to(dev).clone()

  num_gaussians = len(means)
  colors = torch.full((num_gaussians, *color_shape), initial_color, device=dev)

  return create_gaussians(means, scales, quats, opacities, colors)


In [ ]:
from gsplat import rasterization

def render_gaussians(gaussians, viewmat, K, width, height, render_mode="RGB", sh_degree=None, color_activation=None):
  means = gaussians["means"]
  quats = gaussians["quats"]
  scales = torch.exp(gaussians["scales"])
  opacities = torch.sigmoid(gaussians["opacities"])
  raw_colors = gaussians["colors"]

  colors = color_activation(raw_colors) if color_activation is not None else raw_colors

  rendered_colors, rendered_alphas, info = rasterization(
      means=means,
      quats=quats,
      scales=scales,
      opacities=opacities,
      colors=colors,
      viewmats=viewmat.unsqueeze(0),
      Ks=K.unsqueeze(0),
      width=width,
      height=height,
      sh_degree=sh_degree,
      render_mode=render_mode,
  )

  return rendered_colors[0], rendered_alphas[0], info


albdo = torch.sigmoid
shdo = F.softplus
resdo = lambda c: c

In [54]:
def step(loss, optimizers, strategy=None, state=None, params=None, step_idx=0, info=None):
    loss.backward()
    if strategy is not None:
        strategy.step_post_backward(
            params=params,
            optimizers=optimizers,
            state=state,
            step=step_idx,
            info=info,
        )
    for optimizer in optimizers.values():
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)


def save_checkpoint(params, checkpoint_directory, scene, predictor):
    os.makedirs(checkpoint_directory, exist_ok=True)
    path = os.path.join(checkpoint_directory, f"{scene}_{predictor}.pt")
    checkpoint = {name: tensor.detach().cpu() for name, tensor in params.items()}
    torch.save(checkpoint, path)
    print("Saved to: " + path)

def load_checkpoint(path):
  return {k: v.to(dev) for k, v in torch.load(path).items()}

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from gsplat import rasterization


def _tensor_gib(t: torch.Tensor) -> float:
  return t.numel() * t.element_size() / (1024**3)


def get_views(paths, root, predictor, dataset, scene, store_device="cpu"):
  images = []
  albedos = []
  depths = []

  for idx, p in enumerate(paths):
    file_name = os.path.basename(p).split(".")[0]

    with Image.open(p) as img:
      width, height = img.size
      rgb = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0

    i = torch.from_numpy(rgb)
    assert i.ndim == 3 and i.shape[2] == 3, (
        f"Expected (H, W, 3) for {p}, got {tuple(i.shape)}"
    )

    albedo_path = os.path.join(
        root, predictor, dataset, scene, "albedos", f"{file_name}.npy"
    )
    albedo = torch.from_numpy(np.asarray(np.load(albedo_path), dtype=np.float32))

    depth_path = os.path.join(
        root, "depth_anything_v2", "depths", scene, dataset, f"{file_name}.png"
    )
    with Image.open(depth_path) as d_img:
      depth_np = np.asarray(d_img, dtype=np.float32)

    depth = torch.from_numpy(depth_np)
    depth_npy_path = os.path.splitext(depth_path)[0] + ".npy"
    np.save(depth_npy_path, depth_np)

    images.append(i.to(store_device))
    albedos.append(albedo.to(store_device))
    depths.append(depth.detach().clone().to(store_device))

  stacked_images = torch.stack(images, dim=0)
  stacked_albedos = torch.stack(albedos, dim=0)
  stacked_depths = torch.stack(depths, dim=0)

  return stacked_images, stacked_albedos, stacked_depths

In [13]:

from scipy.spatial import cKDTree

def initialise_gaussians(
    points,
    init_scale=1.0,
    init_opacity=0.1,
    scene_scale=1.0,
    dev=dev
):
    pts_cpu = np.asarray(points, dtype=np.float32)

    tree = cKDTree(pts_cpu)
    distances, _ = tree.query(pts_cpu, k=4)
    average_distance = distances[:, 1:].mean(axis=-1)
    average_distance = np.maximum(average_distance, 1e-4)
    pts = torch.as_tensor(
        pts_cpu, dtype=torch.float32, device=dev
    )

    average_distance = torch.as_tensor(
        average_distance, dtype=torch.float32, device=dev
    )

    num_pts = len(pts)

    scales = torch.log(
        average_distance * init_scale
    ).unsqueeze(-1).repeat(1, 3)

    quats = F.normalize(
        torch.rand((num_pts, 4), device=dev),
        dim=-1
    )

    opacities = torch.logit(
        torch.full((num_pts,), init_opacity, device=dev)
    )

    colors = torch.zeros((num_pts, 3), device=dev)

    return create_gaussians(
        pts, scales, quats, opacities, colors,
        scene_scale=scene_scale
    )


In [51]:

ROOT = "/content/drive/MyDrive/3dgs/data/mipnerf360/room"
scene, predictor = "room", "rgbx"
CKPT = "/content/drive/MyDrive/3dgs/ckpt"

paths = sorted(get_image_paths(ROOT))[:2]
viewmats, intrinsics, (width, height), points = get_cameras(ROOT, paths)
images, albedos, depths = get_views(paths, "/content/drive/MyDrive/3dgs", predictor, "mipnerf360", scene)

print(len(paths), "views,", width, "x", height, ",", len(points), "points")

print(images.shape, albedos.shape, depths.shape)


2 views, 389 x 259 , 112627 points
torch.Size([2, 259, 389, 3]) torch.Size([2, 259, 389, 3]) torch.Size([2, 259, 389])


In [18]:
!cp -r /root/.cache/torch_extensions /content/drive/MyDrive/3dgs/torch_ext_cache


In [ ]:
!mkdir -p /root/.cache && cp -r /content/drive/MyDrive/3dgs/torch_ext_cache /root/.cache/torch_extensions


In [15]:
def to_dev(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)


A 0: loss=0.6723, albedo=0.6446, depth=0.2771, N=112627, GPU=0.07 GB
Saved to: /content/drive/MyDrive/3dgs/ckpt/albedo/room_rgbx.pt


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


In [ ]:
def fit_albedo(points, viewmats, intrinsics, width, height, images, albedos, depths, scene, predictor, ckpt_dir, iters=7000, lam_depth=0.1):
    viewmats = torch.as_tensor(viewmats, dtype=torch.float32, device=dev)
    intrinsics = torch.as_tensor(intrinsics, dtype=torch.float32, device=dev)
    images = torch.as_tensor(images, dtype=torch.float32, device=dev)
    albedos = torch.as_tensor(albedos, dtype=torch.float32, device=dev)
    depths = torch.as_tensor(depths, dtype=torch.float32, device=dev)

    if albedos.max().item() > 1.5:
        albedos = albedos / 255.0

    params, optimizers = initialise_gaussians(points)
    strategy = DefaultStrategy(verbose=False)
    strategy.check_sanity(params, optimizers)
    state = strategy.initialize_state()

    for it in range(iters):
        v = np.random.randint(len(images))
        target_albedo = albedos[v:v+1].to(dev)
        target_depth = depths[v:v+1].to(dev)
        out, alphas, info = render_gaussians(params, viewmats[v], intrinsics[v], width, height, render_mode="RGB+ED", color_activation=albedo)

        out = out.squeeze(0)
        alphas = alphas.squeeze(0)

        pred = out[..., :3]
        depth = out[..., 3]

        strategy.step_pre_backward(params=params, optimizers=optimizers, state=state, step=it, info=info )

        if it == 0:
          print()

        albedo_loss = photo_loss(pred, target_albedo)

        depth_loss = pearson_loss(1.0 / depth.clamp(min=1e-3), target_depth)

        loss = albedo_loss + lam_depth * depth_loss

        step(loss, optimizers, strategy, state, params, it, info)

        del out, alphas, info
        del pred, depth, loss
        del albedo_loss, depth_loss
        del target_albedo, target_depth

    save_checkpoint(params, os.path.join(ckpt_dir, "albedo"), scene,predictor)

    return params

In [ ]:
def freeze_albedo(viewmats, intrinsics, width, height, images, scene, predictor, ckpt_dir,
                iters=3000, lam_tv=0.05):
    viewmats = torch.as_tensor(viewmats, dtype=torch.float32, device=dev)
    intrinsics = torch.as_tensor(intrinsics, dtype=torch.float32, device=dev)
    images = torch.as_tensor(images, dtype=torch.float32, device=dev)

    if (images.max().item() > 1.5):
      images = images/255.0
    path = os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt")

    albedo_params = load_checkpoint(os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt"))
    params, optimizers = initialise_from_albedos(albedo_params, (3,), initial_color=0.54)

    for it in range(iters):
      v = np.random.randint(len(images))
      target_image = albedos[v:v+1].to(dev)

      with torch.no_grad():
        albedo, albedo_alphas, _ = render_gaussians(albedo_params, viewmats[v], intrinsics[v], width, height, color_activation=albdo)
        albedo = albedo.squeeze(0)

      shading, shading_alphas, info = render_gaussians(params, viewmats[v], intrinsics[v], width, height, color_activation=shdo)
      shading = shading.squeeze(0)

      if it == 0:
            print(albedo.shape, shading.shape, target_image.shape)

      photo = photo_loss(albedo * shading, target_image)
      tv_loss = tv(shading)
      loss = photo + lam_tv * tv_loss
      step(loss, optimizers, step_idx=it)
      if it % 500 == 0:
          print(f"B {it}: loss {loss.item():.4f}")

      del albedo, albedo_alphas, shading, shading_alphas, info
      del photo, tv_loss, loss
      del target_image

    save_checkpoint(
        params,
        os.path.join(ckpt_dir, "shading"),
        scene,
        predictor
    )

    return params

In [ ]:
def freeze_albedo_shading(viewmats, intrinsics, width, height, images, scene, predictor, ckpt_dir,
                iters=3000, lam_res=0.05):
    viewmats = torch.as_tensor(viewmats, dtype=torch.float32, device=dev)
    intrinsics = torch.as_tensor(intrinsics, dtype=torch.float32, device=dev)
    images = torch.as_tensor(images, dtype=torch.float32, device=dev)

    if (images.max().item() > 1.5):
      images = images/255.0
    albedo_path = os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt")
    shading_path = os.path.join(ckpt_dir, "shading", f"{scene}_{predictor}.pt")

    albedo_params = load_checkpoint(os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt"))
    shading_params = load_checkpoint(os.path.join(ckpt_dir, "shading", f"{scene}_{predictor}.pt"))

    params, optimizers = initialise_from_albedos(albedo_params, (9,3), initial_color=0.0)

    for it in range(iters):
      v = np.random.randint(len(images))
      target_image = images[v:v+1].to(dev)

      with torch.no_grad():
        albedo, albedo_alphas, _ = render_gaussians(albedo_params, viewmats[v], intrinsics[v], width, height, color_activation=albdo)
        albedo = albedo.squeeze(0)
        shading, shading_alphas, info = render_gaussians(shading_params, viewmats[v], intrinsics[v], width, height, color_activation=shdo)
        shading = shading.squeeze(0)

      residuals, residuals_alphas, info = render_gaussians(params, viewmats[v], intrinsics[v], width, height, sh_degree=2, color_activation=resdo)
      residuals = residuals.squeeze(0) - SH_OFFSET * residuals_alphas.squeeze(0)


      if it == 0:
            print(albedo.shape, shading.shape, target_image.shape)

      photo = photo_loss(albedo * shading + residuals, target_image)

      res_loss = residuals.abs().mean()

      loss = photo + lam_res * res_loss

      step(loss, optimizers, step_idx=it)

      if it % 500 == 0:
          print(f"C {it}: loss {loss.item():.4f}")

      del albedo, albedo_alphas, shading, shading_alphas, residuals, residuals_alphas, info
      del photo, res_loss, loss
      del target_image


    save_checkpoint(
        params,
        os.path.join(ckpt_dir, "residual"),
        scene,
        predictor
    )

    return params

In [ ]:
albedo_params = fit_albedo(
    points, viewmats, intrinsics, width, height,
    images, albedos, depths,
    scene, predictor, CKPT,
    iters=1
)

shading_params = freeze_albedo(
    viewmats, intrinsics, width, height, images,
    scene, predictor, CKPT,
    iters=1
)

residual_params = freeze_albedo_shading(
    viewmats, intrinsics, width, height, images,
    scene, predictor, CKPT, iters=1
)